In [ ]:
!pip install pandas

## 1. 학기별 개설교과목정보 데이터 합치기

In [ ]:
import os
import glob
import pandas as pd
import re

files = glob.glob("../data/*.xls")

all_data = []

for file in files:
    # HTML 테이블 읽기
    df = pd.read_html(file)[0]
    
    # 첫 번째 행을 컬럼명으로 지정
    df.columns = df.iloc[0]   # 0번 행을 컬럼명으로
    df = df.drop(index=0)     # 0번 행 삭제
    
    # 학기 정보 컬럼 추가
    semester = os.path.basename(file).replace("-sis.xls", "")
    df["개설학기정보"] = semester
    
    all_data.append(df)

# 모든 파일 병합
df_all = pd.concat(all_data, ignore_index=True)

print(df_all.head())


In [ ]:
df_all.head(3)

In [ ]:
df_all.info()

In [ ]:
df_all["개설학기정보"].unique()

## 2. 필요한 열 선택

In [ ]:
def make_major_df(df_all, codes=None, prefix=None):
    if codes is not None:
        df_major = df_all[df_all["과목번호"].isin(codes)]
    elif prefix is not None:
        df_major = df_all[df_all["과목번호"].str.startswith(prefix)]
    else:
        raise ValueError("codes 또는 prefix 중 하나를 지정해야 합니다.")

    result = []
    for code in df_major["과목번호"].unique():
        subset = df_major[df_major["과목번호"] == code]
        sems = ", ".join(sorted(subset["개설학기정보"].unique()))
        names = ", ".join(sorted(subset["과목명"].unique()))
        profs = ", ".join(sorted(subset["교수진"].dropna().unique()))
        result.append({
            "과목번호": code,
            "과목명": names,
            "교수": profs,
            "개설학기": sems
        })
    return pd.DataFrame(result)


## 3. 변수명 만들기 자동화

In [ ]:
def make_major_dfs(df_all, major_name, codes=None, prefix=None):
    import builtins

    # 정리된 전공 데이터프레임 생성
    df_major = make_major_df(df_all, codes=codes, prefix=prefix)
    builtins.__dict__[f"df_{major_name}"] = df_major

    # 계절별 데이터 생성
    builtins.__dict__[f"df_{major_name}_spring"] = df_major[df_major["개설학기"].str.contains("-1", na=False)]
    builtins.__dict__[f"df_{major_name}_fall"] = df_major[df_major["개설학기"].str.contains("-2", na=False)]
    builtins.__dict__[f"df_{major_name}_summer"] = df_major[df_major["개설학기"].str.contains("-summer", case=False, na=False)]
    builtins.__dict__[f"df_{major_name}_winter"] = df_major[df_major["개설학기"].str.contains("-winter", case=False, na=False)]


### 4-1. 빅데이터사이언스 연계전공

In [ ]:
biksa_codes = [
    "STS2011", "MAT2110", "MAT3020", "MGT2002", "ECO2004", 
    "BDS4010", "CSW4010", "CSE4187",
    "AIC4012", "BDS3010", "BDS3020", "CSE4130", "CSW2010", "CSW2020",
    "CSW2030", "CSE3080", "CSW2050", "CSW3010", "CSE3081", "CSW3030", "CSE4110",
    "CSW3060", "CSW3080",
    "CSW4020", "ECO2009", "ECO3022", "ECO3023", "ECO4003", "ECO4004",
    "ECO4032", "EEE4178", "JAS4014", "MAS1004", "MAS2009",
    "MAS2010", "MAT3110", "MAT4331", "MGT4202", "MGT4208", "MGT4226",
    "MGT4515", "MGT4517", "MGT6613", "MGTG613"
]

make_major_dfs(df_all, major_name="biksa", codes=biksa_codes)

In [ ]:
df_biksa_spring

In [ ]:
df_biksa_summer

In [ ]:
df_biksa_fall

### 4-2. 경영

In [ ]:
make_major_dfs(df_all, major_name="mgt", prefix="MGT")

In [ ]:
df_mgt_fall

### 4-3. 국문

In [ ]:
make_major_dfs(df_all, major_name="kor", prefix="KOR")

In [ ]:
df_kor_fall

### 4-4. 공공인재 연계전공

In [ ]:
pub_codes = [
    "PUB2005", "POL3130", "POL2002", "SOC2001", "SOC2003", "SOC3010",
    "ECO2001", "ECO2002", "ECO3009", "ECO3011", "ECO3017", "MGT2002", 
    "MGT2003", "PHI2005", "PSY2001", "PSY3009", "PUB3030", "PUB3016",
    "PUB3029", "PUB3023", "PUB3024", "PUB3025", "PUB3031", "PUB3032",
    "PUB3026", "PUB3021", "PUB3020", "PUB3022", "PUB3028", "PUB3027",
    "PUB4009", "KOR4500", "EDU2001", "PHI4010", "ECO2007", "MGT3004",
    "MGT4301", "MGT3005", "MGT4404"
]

make_major_dfs(df_all, major_name="pub", codes=pub_codes)


### 4-5. 교직

In [ ]:
edu_codes = [
    "EDU2001", "EDU2002", "EDU2003", "EDU2004", "EDU3001", "EDU3047",
    "EDU3002", "EDU3033", "EDU3045", "EDU3046", "EDU3037", "EDU3004", "EDU3035",
    "EDU2005", "EDU2007", "EDU3038", "EDU3039", "EDU3048", "EDU3049",
    "SHU4019", "SHU4022", "SHU4031", "EDU3036"
]

make_major_dfs(df_all, major_name="edu", codes=edu_codes)


## 5. 이번학기 시간표 만드는 함수

In [ ]:
def make_timetable(df_all, df_major_fall, taken_courses):
    if '개설학기' not in df_all.columns and '개설학기정보' in df_all.columns:
        df_all = df_all.rename(columns={'개설학기정보': '개설학기'})

    if '개설학기' not in df_major_fall.columns and '개설학기정보' in df_major_fall.columns:
        df_major_fall = df_major_fall.rename(columns={'개설학기정보': '개설학기'})

    courses_2025_2 = df_major_fall[df_major_fall["개설학기"].str.contains("2025-2", na=False)]

    merged = df_all[
        (df_all["과목번호"].isin(courses_2025_2["과목번호"])) &
        (df_all["개설학기"] == "2025-2")
    ][["과목번호", "과목명", "수업시간/강의실", "교수진"]]

    # 긴 시간대 쪼개기 매핑
    split_times = {
        "09:00~11:45": ["09:00~10:15", "10:30~11:45"],
        "10:30~13:15": ["10:30~11:45", "12:00~13:15"],
        "13:30~16:15": ["13:30~14:45", "15:00~16:15"]
    }

    expanded_rows = []

    # 이미 수강한 과목 제외
    filtered_df = merged[~merged["과목명"].isin(taken_courses)]

    for _, row in filtered_df.iterrows():
        # NaN 방지 처리
        time_room_str = str(row["수업시간/강의실"]) if pd.notna(row["수업시간/강의실"]) else ""

        # 요일 여러 개 추출
        days = re.findall(r"[월화수목금토일]", time_room_str)
        if not days:
            continue

        # 시간 추출
        time_match = re.search(r"\d{2}:\d{2}~\d{2}:\d{2}", time_room_str)
        if not time_match:
            continue

        time = time_match.group()

        # 긴 시간대 쪼개기
        if time in split_times:
            times_to_add = split_times[time]
        else:
            times_to_add = [time]

        # 요일·시간 모두 확장
        for t in times_to_add:
            for day in days:
                expanded_rows.append({
                    "시간": t,
                    "요일": day,
                    "과목(교수)": f"{row['과목명']}({row['교수진']})"
                })

    # DataFrame 생성
    expanded_df = pd.DataFrame(expanded_rows)

    # 요일·시간별 과목 합치기
    timetable = (
        expanded_df.groupby(["시간", "요일"])["과목(교수)"]
        .apply(lambda x: ", ".join(sorted(set(x))))
        .reset_index()
    )

    # 요일 순서 맞추기
    요일순서 = ["월", "화", "수", "목", "금"]
    timetable_pivot = timetable.pivot(index="시간", columns="요일", values="과목(교수)").fillna("")
    timetable_pivot = timetable_pivot.reindex(columns=요일순서)

    return timetable_pivot


### 5-1. 사용

In [ ]:
hs_taken_courses = [
    "선형대수학", "경영통계학", "경제통계학", "통계학입문", 
    "기초빅데이터프로그래밍", "고급응용C프로그래밍", "인공지능(딥러닝)개론", 
    "웹데이터수집과 텍스트분석(캡스톤디자인)", "Data&AI",
    "회계학원론", "조직행동이론", "마케팅원론",
    "국제경영론", "경영전략",
    "의사결정론"
]

In [ ]:
# 내가 포함하고 싶은 경영 과목명 리스트
selected_mgt_courses = [
    "운영관리",
    "재무관리",
    "경영정보시스템",
    "관리회계",
    "기업윤리",
    "경영과학",
    "경영 데이터사이언스"
]

# 경영 과목 필터링
df_mgt_selected = df_mgt_fall[df_mgt_fall["과목명"].isin(selected_mgt_courses)]


In [ ]:
# 두 전공 과목 목록 합치기
df_combined_fall = pd.concat([df_biksa_fall, df_mgt_selected], ignore_index=True)

# 합친 전공 과목으로 시간표 생성
timetable_hs = make_timetable(df_all, df_combined_fall, hs_taken_courses)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

timetable_hs

In [ ]:
import re

# 1. 매핑 만들 때 과목명 전처리
def clean_course_name(name):
    # 괄호 안 내용 제거
    return re.sub(r"\(.*?\)", "", str(name)).strip()

df_all['과목명_clean'] = df_all['과목명'].apply(clean_course_name)
name_to_code = dict(zip(df_all['과목명_clean'], df_all['과목번호']))

# 2. 색칠 함수
def colorize_cell(cell):
    if pd.isna(cell) or cell.strip() == "":
        return cell

    courses = str(cell).split(', ')
    colored_courses = []

    for course_name in courses:
        # 동일하게 전처리
        course_name_clean = clean_course_name(course_name)

        code = name_to_code.get(course_name_clean, None)
        in_biksa = code in biksa_codes_set if code else False
        in_mgt = code in mgt_codes_set if code else False

        if in_biksa and in_mgt:
            colored_courses.append(f'<span style="color:orange">{course_name}</span>')
        elif in_mgt:
            colored_courses.append(f'<span style="color:red">{course_name}</span>')
        elif in_biksa:
            colored_courses.append(f'<span style="color:blue">{course_name}</span>')
        else:
            colored_courses.append(course_name)

    return "<br>".join(colored_courses)

# 3. 적용
timetable_colored = timetable_hs.map(colorize_cell)
from IPython.display import HTML
HTML(timetable_colored.to_html(escape=False))


In [ ]:
timetable_kor = make_timetable(df_all, df_kor_fall, hs_taken_courses)

timetable_kor

In [ ]:
ssong_taken_courses = [
    "문학이란무엇인가", "국어학입문", "국문학개설",
    "국어음운론", "국어사", "국어의미론",
    "현대시텍스트읽기I",
    "설화 문학의 이해",
    "문학과 융합적 상상력"
]

In [ ]:
timetable_kor_ssong = make_timetable(df_all, df_kor_fall, ssong_taken_courses)

timetable_kor_ssong

In [ ]:
timetable_mgt_ssong = make_timetable(df_all, df_mgt_fall, ssong_taken_courses)
timetable_mgt_ssong

#### 혜주

In [ ]:
hyeju_taken_courses = [
    "회계학원론",
    "마케팅원론", "조직행동이론", "경영통계학"
]

In [ ]:
timetable_kor_hyeju = make_timetable(df_all, df_mgt_fall, hyeju_taken_courses)

timetable_kor_hyeju

### 한나

In [ ]:
hanna_taken_courses = [
    "문예창작론",
    "국제인권과법",
    "고전문학자료연구",
    "디지털교육",
    "세법",
    "인문세미나",
    "교직실무",
    "형법",
    "경제학원론I",
    "프랑스언어와문화II",
    "교육행정및교육경영",
    "생명과환경",
    "문학과융합적상상력",
    "시쓰기",
    "교육방법및교육공학",
    "국어형태론",
    "문학이란무엇인가",
    "문학과문화",
    "디지털인문학강독",
    "스포츠소비자행동론",
    "국어의미론",
    "인문사회글쓰기",
    "기초인공지능프로그래밍",
    "고전소설텍스트읽기",
    "프랑스언어와문화I",
    "국어방언론",
    "현대소설론",
    "테니스",
    "현대세계와윤리문제",
    "국문학개설",
    "대학수학",
    "국어학입문",
    "파워요가",
    "그리스도교윤리",
    "국문말씨"
]


In [ ]:
import re

# 1. 과목명 전처리 (괄호 제거)
def clean_course_name(name):
    return re.sub(r"\(.*?\)", "", str(name)).strip()

df_all['과목명_clean'] = df_all['과목명'].apply(clean_course_name)
name_to_code = dict(zip(df_all['과목명_clean'], df_all['과목번호']))

# 2. 전공별 과목번호 set 만들기
kor_codes_set = set(df_all.loc[df_all['과목번호'].str.startswith("KOR"), '과목번호'])
edu_codes_set = set(edu_codes)
pub_codes_set = set(pub_codes)

df_kor_selected = df_all[df_all['과목번호'].str.startswith("KOR")]
df_edu_selected = df_all[df_all['과목번호'].isin(edu_codes)]
df_pub_selected = df_all[df_all['과목번호'].isin(pub_codes)]

# 세 전공 과목 합치기
df_combined_hanna = pd.concat(
    [df_kor_selected, df_edu_selected, df_pub_selected],
    ignore_index=True
)

# Hanna 시간표 생성
timetable_hanna = make_timetable(
    df_all,
    df_combined_hanna,
    hanna_taken_courses
)
timetable_hanna

# 3. 색칠 함수
def colorize_cell(cell):
    if pd.isna(cell) or cell.strip() == "":
        return cell

    courses = str(cell).split(', ')
    colored_courses = []

    for course_name in courses:
        course_name_clean = clean_course_name(course_name)
        code = name_to_code.get(course_name_clean, None)

        in_kor = code in kor_codes_set if code else False
        in_edu = code in edu_codes_set if code else False
        in_pub = code in pub_codes_set if code else False

        # 두 개 이상 전공에 해당하면 주황색
        if sum([in_kor, in_edu, in_pub]) > 1:
            colored_courses.append(f'<span style="color:orange">{course_name}</span>')
        elif in_kor:
            colored_courses.append(f'<span style="color:red">{course_name}</span>')
        elif in_edu:
            colored_courses.append(f'<span style="color:blue">{course_name}</span>')
        elif in_pub:
            colored_courses.append(f'<span style="color:green">{course_name}</span>')
        else:
            colored_courses.append(course_name)

    return "<br>".join(colored_courses)

# 4. 적용
timetable_colored = timetable_hanna.map(colorize_cell)

from IPython.display import HTML
HTML(timetable_colored.to_html(escape=False))

